In [ ]:
from jax import config
config.update("jax_enable_x64", True)

import numpy as np
import jax
import jax.numpy as jnp
import jax.random as jr
import torch

from sigkernel import SigKernel, LinearKernel

# --------------------------------------------------
# config
# --------------------------------------------------
SEED = 2404029296
DIM = 2
N_PAIRS = 100
HORIZON = 1.0

# sweep only the knobs that plausibly matter
N_FINE_POWS = [12, 13]          # 2^12 is the paper setting; 2^13 tests finer ref
DYADIC_ORDERS = [0, 1, 2]       # naive solver with refinement
X_POINTS = [1, 2]               # only calibrate first two points

torch.set_default_dtype(torch.float64)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def jax_to_torch_f64(x, *, device=DEVICE):
    x_np = np.asarray(jax.device_get(x), dtype=np.float64).copy()
    x_t = torch.from_numpy(x_np).to(device=device, dtype=torch.float64)
    assert x_t.dtype == torch.float64
    return x_t

def torch_to_numpy_f64(x):
    x_np = x.detach().cpu().numpy()
    if x_np.dtype != np.float64:
        x_np = x_np.astype(np.float64, copy=False)
    return x_np

def sample_bm_increments(key, *, n_pairs, n_steps, dim, horizon=1.0):
    dt = horizon / n_steps
    kx, ky = jr.split(key)
    dX = jnp.sqrt(dt) * jr.normal(kx, (n_pairs, n_steps, dim), dtype=jnp.float64)
    dY = jnp.sqrt(dt) * jr.normal(ky, (n_pairs, n_steps, dim), dtype=jnp.float64)
    return dX, dY

def aggregate_increments(dX, block):
    n_pairs, n_steps, dim = dX.shape
    assert n_steps % block == 0
    dXr = dX.reshape(n_pairs, n_steps // block, block, dim)
    return dXr.sum(axis=2)

def increments_to_paths(dX):
    x0 = jnp.zeros((dX.shape[0], 1, dX.shape[2]), dtype=jnp.float64)
    return jnp.concatenate([x0, jnp.cumsum(dX, axis=1)], axis=1)

def compute_naive_vals(paths_x, paths_y, *, dyadic_order, max_batch=5):
    ker = SigKernel(
        LinearKernel(),
        dyadic_order=dyadic_order,
        _naive_solver=True,
    )
    X_t = jax_to_torch_f64(paths_x)
    Y_t = jax_to_torch_f64(paths_y)
    out_t = ker.compute_kernel(X_t, Y_t, max_batch=max_batch)
    assert out_t.dtype == torch.float64
    return torch_to_numpy_f64(out_t)

# --------------------------------------------------
# sample once on the finest grid we need, then coarsen
# --------------------------------------------------
MAX_FINE_POW = max(N_FINE_POWS)
N_MAX = 2**MAX_FINE_POW

key = jr.PRNGKey(SEED)
dX_max, dY_max = sample_bm_increments(
    key,
    n_pairs=N_PAIRS,
    n_steps=N_MAX,
    dim=DIM,
    horizon=HORIZON,
)

results = []

for n_pow in N_FINE_POWS:
    N_FINE = 2**n_pow
    base_block = 2**(MAX_FINE_POW - n_pow)

    dX_fine = aggregate_increments(dX_max, base_block)
    dY_fine = aggregate_increments(dY_max, base_block)

    X_fine = increments_to_paths(dX_fine)
    Y_fine = increments_to_paths(dY_fine)

    for lam in DYADIC_ORDERS:
        fine_vals = compute_naive_vals(X_fine, Y_fine, dyadic_order=lam)

        row = {
            "N_FINE": N_FINE,
            "dyadic_order": lam,
        }

        for x in X_POINTS:
            # x = -log2(mesh size), so coarse block = 2^(n_pow - x)
            coarse_block = 2**(n_pow - x)

            dX_coarse = aggregate_increments(dX_fine, coarse_block)
            dY_coarse = aggregate_increments(dY_fine, coarse_block)

            X_coarse = increments_to_paths(dX_coarse)
            Y_coarse = increments_to_paths(dY_coarse)

            coarse_vals = compute_naive_vals(X_coarse, Y_coarse, dyadic_order=lam)
            err = np.mean(np.abs(coarse_vals - fine_vals))
            y = -np.log10(err)

            row[f"y_at_x={x}"] = float(y)

        results.append(row)

# sort by how close the first two points are to the paper visually
TARGET_X1 = 0.85
TARGET_X2 = 0.95

for row in results:
    row["score_vs_visual_target"] = (
        (row["y_at_x=1"] - TARGET_X1)**2 +
        (row["y_at_x=2"] - TARGET_X2)**2
    )

results = sorted(results, key=lambda r: r["score_vs_visual_target"])

print("Best candidates:")
for row in results:
    print(
        f"N_FINE={row['N_FINE']}, "
        f"lambda={row['dyadic_order']}, "
        f"x=1 -> {row['y_at_x=1']:.6f}, "
        f"x=2 -> {row['y_at_x=2']:.6f}, "
        f"score={row['score_vs_visual_target']:.6f}"
    )